## Simple Web Query
Here we can use the Python SDK to develop the simple web query agent, then save the agent to a config.yaml and run it from there.

In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [ ]:
from examples.getting_started.simple_web_query.src.nat_simple_web_query.register import WebQueryTool
from nat.data_models.component_ref import EmbedderRef
from nat.embedder.nim_embedder import NIMEmbedder
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_agent import NatReactAgent

llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    name="nv-embedqa-e5-v5",
)

current_time_tool = CurrentTimeTool(
    name="current_datetime",
)
web_query_tool = WebQueryTool(
    webpage_url="https://docs.smith.langchain.com",
    description="Search for information about LangSmith. For any questions about LangSmith, you must use this tool!",  # noqa: E501
    embedder_name=EmbedderRef(embedder.name),
    chunk_size=512,
    name="custom_webpage_query",
)

agent = NatReactAgent(
    tools=[web_query_tool, current_time_tool],
    llm=llm,
    referenced_embedders=[embedder],
    verbose=True,
    parse_agent_response_max_retries=3,
)

In [ ]:
await agent.prompt('What is LangSmith?')

In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())